# 12 · 프레임워크 통합 (DLPack · PyTorch 무복사 연동)

> **CuPy 2일 집중 코스 — Day 2 / 단원 9 (실전 예제)**

CuPy 배열을 **복사 없이(zero-copy)** PyTorch 등 다른 GPU 라이브러리와 주고받는 법을 배웁니다.
같은 GPU 메모리를 공유하므로 전처리는 CuPy로, 모델은 PyTorch로 — 전송 없이 연결할 수 있습니다.

## 학습 목표
- **DLPack** 표준과 `__cuda_array_interface__`로 무복사 교환을 이해한다.
- CuPy ↔ PyTorch 텐서를 **포인터 공유**로 변환한다.
- 소유권·동기화 주의점을 안다.

## 목차
1. [왜 상호운용인가](#1) · 2. [DLPack 표준](#2) · 3. [CuPy → PyTorch](#3)
4. [PyTorch → CuPy](#4) · 5. [`__cuda_array_interface__`](#5) · 6. [주의점](#6) · 7. [연습 & 체크포인트](#7)

> `torch`(CUDA 빌드)가 있으면 실습이 동작합니다. 없으면 안내만 출력하고 개념은 그대로 학습합니다.

In [ ]:
import numpy as np, cupy as cp
from course_utils import print_env, allclose
print_env()
try:
    import torch
    HAS_TORCH = torch.cuda.is_available()
    print('torch CUDA 사용 가능:', HAS_TORCH)
except Exception as e:
    HAS_TORCH = False
    print('torch 미설치 — 개념만 학습합니다. (', e, ')')

<a id="1"></a>
## 1. 왜 상호운용인가

딥러닝 파이프라인은 보통 **전처리(CuPy/cuDF) → 모델(PyTorch/TF)** 로 이어집니다.
- 라이브러리마다 따로 GPU 배열을 복사하면 **전송 낭비**
- 같은 GPU 메모리를 **포인터로 공유**하면 복사 0 → 빠름
- 표준: **DLPack**, **CUDA Array Interface**(`__cuda_array_interface__`)

<a id="2"></a>
## 2. DLPack 표준

**DLPack**은 프레임워크 간 텐서를 무복사로 교환하는 표준입니다. 최신 API는 `from_dlpack()` 하나로 통일됩니다
(객체가 `__dlpack__`를 구현하면 됨). 구버전은 `toDlpack()`/`fromDlpack(capsule)`을 씁니다.

In [ ]:
# CuPy 내부 DLPack 왕복 (포인터 동일성 확인)
x = cp.arange(10, dtype=cp.float32)
y = cp.from_dlpack(x)          # 무복사
print('같은 메모리?', x.data.ptr == y.data.ptr)   # True

<a id="3"></a>
## 3. CuPy → PyTorch (무복사)

`torch.from_dlpack(cupy_array)` 로 CuPy 배열을 PyTorch 텐서로 (복사 없이) 봅니다.

In [ ]:
x = cp.arange(1_000_000, dtype=cp.float32)
if HAS_TORCH:
    t = torch.from_dlpack(x)              # CuPy -> torch (zero-copy)
    print('torch device:', t.device, '| 같은 포인터?', t.data_ptr() == x.data.ptr)
    t += 1                                # torch에서 수정하면
    print('CuPy에도 반영?', float(x[0]))  # 공유 메모리라 반영됨
else:
    print('torch 없음 — 개념: torch.from_dlpack(cupy_array)')

<a id="4"></a>
## 4. PyTorch → CuPy (무복사)

반대로 `cp.from_dlpack(torch_tensor)` 로 PyTorch 텐서를 CuPy 배열로 봅니다.

In [ ]:
if HAS_TORCH:
    t = torch.ones(1_000_000, device='cuda', dtype=torch.float32)
    g = cp.from_dlpack(t)                 # torch -> CuPy (zero-copy)
    print('같은 포인터?', g.data.ptr == t.data_ptr())
    g *= 2                                 # CuPy에서 수정
    print('torch에도 반영?', float(t[0]))
else:
    print('torch 없음 — 개념: cp.from_dlpack(torch_tensor)')

<a id="5"></a>
## 5. `__cuda_array_interface__`

CuPy 배열은 `__cuda_array_interface__`(CAI) 속성으로 **GPU 포인터·shape·dtype·strides**를 노출합니다.
Numba·PyTorch 등 CAI를 이해하는 라이브러리는 이를 보고 무복사로 접근합니다.

In [ ]:
x = cp.arange(6, dtype=cp.float32).reshape(2,3)
cai = x.__cuda_array_interface__
print('shape:', cai['shape'], '| typestr:', cai['typestr'])
print('data ptr:', cai['data'][0] == x.data.ptr)   # 디바이스 포인터

<a id="6"></a>
## 6. 주의점

- **소유권/수명**: 원본 배열이 살아 있어야 공유 뷰가 유효(원본이 해제되면 위험)
- **동기화/스트림**: 서로 다른 스트림에서 같은 메모리를 쓰면 이벤트로 순서 보장 필요(06)
- **dtype/연속성**: 일부 변환은 연속(C-contiguous)·지원 dtype을 요구
- 무복사는 **같은 디바이스**에서만 — 다른 GPU면 전송 필요

<a id="7"></a>
## 7. 연습 — CuPy 전처리 → PyTorch → CuPy (무복사)

CuPy로 표준화한 뒤 **무복사로 PyTorch에 넘겨** 연산하고, 다시 CuPy로 받아 후처리하는 함수를 완성하세요(전송 0).

In [ ]:
def pipeline(x_cp):
    # 1) CuPy 전처리: 표준화
    z = (x_cp - x_cp.mean()) / (x_cp.std() + 1e-6)
    if not HAS_TORCH:
        return z
    # TODO: t = torch.from_dlpack(z); t = torch.relu(t)   # 무복사 + torch 연산
    # TODO: return cp.from_dlpack(t)                       # 다시 CuPy로
    raise NotImplementedError

x = cp.random.randn(1_000_000, dtype=cp.float32)
# out = pipeline(x); print(type(out))

<details><summary>💡 해답 보기</summary>

```python
def pipeline(x_cp):
    z = (x_cp - x_cp.mean()) / (x_cp.std() + 1e-6)
    if not HAS_TORCH:
        return cp.maximum(z, 0)
    t = torch.from_dlpack(z)      # zero-copy
    t = torch.relu(t)
    return cp.from_dlpack(t)      # zero-copy back

x = cp.random.randn(1_000_000, dtype=cp.float32)
out = pipeline(x)
# 검증: relu(표준화) 와 일치
z = (x - x.mean())/(x.std()+1e-6)
allclose(cp.maximum(z,0), out, rtol=1e-4, atol=1e-4, name='interop pipeline')
```
</details>

<a id="8"></a>
## 8. Numba ↔ CuPy & 추가 주제

**Numba CUDA 커널은 CuPy 배열을 직접 받습니다** — CuPy가 `__cuda_array_interface__`(CAI)를 노출하므로 무복사로 접근합니다.
즉 08~09에서 만든 Numba 커널을 CuPy 배열에 바로 적용할 수 있습니다.

In [ ]:
try:
    from numba import cuda
    @cuda.jit
    def add_one(a):
        i = cuda.grid(1)
        if i < a.size: a[i] += 1.0
    g = cp.arange(16, dtype=cp.float32)
    add_one[1, 16](g)            # Numba가 CuPy 배열을 CAI로 직접 수정(무복사·in-place)
    print(cp.asnumpy(g))         # 1..16
except Exception as e:
    print('numba 미설치:', e)

**2D·dtype 주의**: 무복사 공유는 보통 **연속(C-contiguous)·지원 dtype**을 요구합니다. 
GPU 메모리는 본질적으로 1차원의 긴 선형 구조입니다. 2D 이상의 다차원 배열을 이 1차원 메모리에 어떻게 구겨 넣느냐에 따라 연속성(Contiguity)이 결정됩니다.
- C-Contiguous (행 기준 연속)란?
  * 기본적으로 NumPy나 CuPy에서 2D 배열을 생성하면 C-Contiguous (C-order) 방식으로 메모리에 저장됩니다. 이는 같은 행(Row)에 있는 데이터가 메모리상에 나란히(연속적으로) 배치된다는 뜻입니다.

- 비연속 데이터가 발생하는 경우
    * 배열을 생성할 때는 연속적이지만, 배열을 조작하다 보면 메모리는 그대로인데 뷰(View)만 바뀌면서 비연속 데이터가 됩니다.
    * 전치 (Transpose): arr.T를 호출하면 행과 열을 읽는 순서만 바뀔 뿐 메모리는 재배치되지 않습니다. (C-order가 F-order로 변환됨)
    * 슬라이싱 (Slicing): arr[:, 1:3]처럼 특정 열만 추출하면, 메모리상에서는 듬성듬성 떨어진 데이터를 건너뛰며(Stride) 읽어야 합니다.

- 왜 Numba에서 문제가 될까?
    * CuPy 배열이 비연속 상태일 때 이를 Numba 커널에 넘기면 두 가지 문제가 발생할 수 있습니다.
       * 데이터 오염 및 충돌: Numba 커널이 데이터의 보폭(Stride) 정보를 완벽하게 해석하지 못하거나 무시하고 선형적으로 메모리에 접근하면, 전혀 엉뚱한 값을 읽거나 쓰게 됩니다.
       * 성능 저하 (Memory Coalescing 실패): GPU는 여러 스레드가 한 번에 인접한 메모리를 덩어리째 읽어오는(Coalesced Memory Access) 방식으로 속도를 냅니다. 데이터가 듬성듬성 떨어져 있으면 메모리 접근 횟수가 급증하여 성능이 폭락합니다.
    * 해결책: cp.ascontiguousarray()
       * 비연속 배열을 Numba 커널에 넘기기 전에는 반드시 메모리를 연속된 공간에 새로 복사하여 재배열해야 합니다.

- 스트림(Stream)과 이벤트(Event) 동기화
    * GPU는 여러 작업을 동시에 처리할 수 있는 비동기(Asynchronous) 장치입니다. 
    * 스트림(Stream)은 GPU에 내리는 명령(데이터 복사, 커널 실행 등)의 대기열(Queue)입니다.
    * 스트림 분리로 인한 레이스 컨디션(Race Condition)
       * 같은 스트림에 들어간 명령은 순서대로 실행되지만, 서로 다른 스트림에 들어간 명령은 GPU가 동시에 병렬로 실행합니다.
       * CuPy 작업과 Numba CUDA 커널이 서로 다른 스트림을 사용하도록 설정되어 있다면 심각한 문제가 발생할 수 있습니다.
           * 상황: CuPy로 A 배열에 연산을 수행한 뒤, 그 결과를 Numba 커널이 받아서 B 배열에 쓴다고 가정해 봅시다.
           * 문제: CuPy가 A 배열에 값을 다 쓰기도 전에, Numba 커널이 동시에 실행되어 아직 계산되지 않은 쓰레기 값을 읽어갈 수 있습니다.
    * 이벤트(Event)를 이용한 순서 보장
       * 이러한 충돌을 막기 위해 두 스트림 간에 "이 작업이 끝날 때까지 기다려"라는 신호등을 세워야 합니다. 이때 사용하는 것이 CUDA 이벤트(Event)입니다.
           * CuPy 연산이 끝나는 지점에 이벤트(Event)를 기록(Record)합니다.
           * Numba 커널이 실행될 스트림에게 해당 이벤트가 완료될 때까지 대기(Wait)하라고 지시합니다.



**연습 — 무복사 왕복 & 소유권 확인**: CuPy 배열을 (있으면)torch로 무복사 변환→torch에서 수정→**원본 CuPy에도 반영**되는지 확인하세요.

In [ ]:
x = cp.zeros(8, dtype=cp.float32)
# TODO: HAS_TORCH면 t=torch.from_dlpack(x); t.add_(5); print(cp.asnumpy(x))  # 5로 채워졌나?
# 같은 메모리를 공유하므로 torch 수정이 CuPy에 보여야 함

<details><summary>💡 해답 보기</summary>

```python
if HAS_TORCH:
    t = torch.from_dlpack(x)   # zero-copy view
    t.add_(5)                  # torch에서 in-place 수정
    print(cp.asnumpy(x))       # [5,5,...] — 공유 메모리라 반영
    print('포인터 동일?', t.data_ptr() == x.data.ptr)
else:
    print('torch 없음 — 개념만: from_dlpack은 메모리를 공유한다')
```
</details>

### 체크포인트
- [ ] DLPack `from_dlpack`로 무복사 교환을 이해했다
- [ ] CuPy↔PyTorch를 포인터 공유로 변환했다
- [ ] `__cuda_array_interface__`의 역할을 안다
- [ ] 소유권·동기화 주의점을 안다

다음: **`13_dl_preprocess_capstone`** — Day 2 종합 캡스톤(커스텀 커널 + interop).